In [1]:
import subprocess
import sys

# # List of required packages
# required_packages = [
#     'opencv-python',
#     'mediapipe',
#     'pandas',
#     'numpy',
#     'tqdm',
#     'scikit-learn',
#     'mord',
#     'tensorflow',
#     'matplotlib',
#     'tensorflow-addons',
#     'imbalanced-learn'
# ]

# # Function to install packages if not already installed
# def install_package(package):
#     subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# # Check and install missing packages
# for package in required_packages:
#     try:
#         __import__(package)
#     except ImportError:
#         print(f"{package} not found, installing...")
#         install_package(package)

# Now, import the libraries

import os
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
from tqdm import tqdm
from multiprocessing import Pool
import logging
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from mord import LogisticAT  # Ordinal logistic regression
from sklearn.metrics import classification_report, confusion_matrix, mean_absolute_error#, root_mean_squared_error
import tensorflow as tf
from tensorflow import keras
from keras import layers
import matplotlib.pyplot as plt
from keras import backend as K
from keras import losses
from keras.layers import Lambda
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from keras.preprocessing.sequence import pad_sequences
import random
import concurrent.futures
from tqdm import tqdm
from datetime import datetime
import json
from sklearn.decomposition import PCA


# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("process_video.log"),
        logging.StreamHandler()
    ]
)

In [2]:
def plot(df):
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    import matplotlib.gridspec as gridspec

    # Load your dataset
    # df = pd.read_csv('your_dataset.csv')  # Uncomment if needed

    # Unique classes and colors
    classes = sorted(df['target'].unique())
    cmap = plt.get_cmap('tab10')
    colors = [cmap(i) for i in range(len(classes))]

    # Create figure and layout
    fig = plt.figure(figsize=(10, 10))
    gs = gridspec.GridSpec(4, 4)

    # Scatter plot (main panel)
    ax_main = fig.add_subplot(gs[1:4, 0:3])

    # Top distribution (PC1)
    ax_xdist = fig.add_subplot(gs[0, 0:3], sharex=ax_main)

    # Right distribution (PC2)
    ax_ydist = fig.add_subplot(gs[1:4, 3], sharey=ax_main)

    # Plot scatter for each class
    for i, cls in enumerate(classes):
        cls_data = df[df['target'] == cls]
        
        # Scatter
        ax_main.scatter(
            cls_data['PC1'], cls_data['PC2'],
            color=colors[i],
            alpha=0.6,
            edgecolors='k',
            label=f'Class {cls}',
            s=30
        )
        
        # KDE for PC1 (top)
        sns.kdeplot(cls_data['PC1'], ax=ax_xdist, color=colors[i], label=f'Class {cls}', fill=True, alpha=0.3)
        
        # KDE for PC2 (right)
        sns.kdeplot(cls_data['PC2'], ax=ax_ydist, color=colors[i], fill=True, alpha=0.3, vertical=True)

    # Hide x-axis labels on top and y-axis labels on right
    ax_xdist.set_ylabel('Density')
    ax_ydist.set_xlabel('Density')
    
    # Create a custom legend in the top-right corner of the figure
    handles, labels = ax_main.get_legend_handles_labels()
    fig.legend(handles, labels, title='Target Class', loc='upper right', bbox_to_anchor=(0.98, 0.98))

    ax_xdist.grid(True)
    ax_ydist.grid(True)

    # Set main scatter plot labels
    ax_main.set_xlabel('PC1')
    ax_main.set_ylabel('PC2')
    ax_main.grid(True)
    ax_main.set_title('PC1 vs PC2 Scatter Plot with Class Distributions')

    # Tidy layout
    plt.tight_layout()
    plt.savefig('pc1_pc2_class_distribution_plot.png', dpi=300)
    plt.close()



# Function to set the random seed
def set_seed(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    tf.config.experimental.enable_op_determinism()


def reshape_data_rnn(df, feature_columns=[], max_slices=30, sortby_column='slice_index', target_column='target'):
    X, y = [], []
    
    grouped = df.groupby('filename')
    
    for video, group in grouped:
        # Sort by slice_index to maintain temporal order
        group = group.sort_values(by=sortby_column)

        # Extract features
        slices = group[feature_columns].values.tolist()

        # Split into chunks of max_slices
        for i in range(0, len(slices), max_slices):
            chunk = slices[i:i + max_slices]
            X.append(chunk)
            y.append(group[target_column].iloc[0])  # Same label for all chunks from the same video

    # Pad sequences to the max_slices length
    X_padded = pad_sequences(X, maxlen=max_slices, padding='post', dtype='float32')

    # Convert labels to numpy array
    y = np.array(y)
    
    return X_padded, y



def report(model, x, y_true, unique_classes):
    y_true = np.array(y_true)
    y_pred = np.array(model.predict(x, verbose=0)).flatten()
    results = {}
    for cls in unique_classes:
        idx = y_true == cls
        if np.all(idx == False):
            results[cls] = -1
            continue
        mae = mean_absolute_error(y_true[idx], y_pred[idx])
        results[cls] = mae

    return y_pred, results


# def weighted_ordinal_mse(y_true, y_pred):
#     y_true = K.round(y_true)
#     y_pred = K.round(y_pred)
#     y_pred = tf.cast(y_pred, tf.int64) # Cast y_pred to int64
#     error = K.abs(y_true - y_pred)
#     weights = K.square(error)
#     return K.mean(K.square(tf.cast(y_true, tf.float32) - tf.cast(y_pred, tf.float32)) * tf.cast(weights, tf.float32))

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.integer)):
            return int(obj)
        elif isinstance(obj, (np.floating)):
            return round(float(obj), 1)
        elif isinstance(obj, (np.ndarray)):
            return [round(float(x), 1) if isinstance(x, (np.floating, float)) else x for x in obj.tolist()]
        return super(NumpyEncoder, self).default(obj)


In [3]:
class Model:
    def __init__(self, units, dropout_ratio, kernel_regularizer_l2, kernel_regularizer_l1, batch_size, num_classes, learning_rate, weight_decay, beta_1, beta_2, epsilon, epochs, patience, max_slices, mapping, dataset_name):
        self.units = units
        self.dropout_ratio = dropout_ratio
        self.kernel_regularizer_l2 = kernel_regularizer_l2
        self.kernel_regularizer_l1 = kernel_regularizer_l1
        self.batch_size = batch_size
        self.num_classes = num_classes
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.beta_1 = beta_1
        self.beta_2 = beta_2
        self.epsilon = epsilon
        self.epochs = epochs
        self.patience = patience
        self.max_slices = max_slices
        self.mapping = mapping
        self.dataset_name = dataset_name
        self.dataset_id = os.path.splitext(dataset_name)
        # self.models = []
        # self.histories = []
        # self.reports = []
        
        
    def build_model(self, input_shape):
        model = keras.Sequential()
        model.add(layers.Input(shape=(input_shape[0], input_shape[1])))

        # Add LSTM layers
        for u in self.units[:-1]:
            model.add(layers.LSTM(u, activation='tanh', return_sequences=True, 
                                  kernel_regularizer=tf.keras.regularizers.l2(self.kernel_regularizer_l2)))
            model.add(layers.BatchNormalization())
            model.add(layers.Dropout(self.dropout_ratio))

        # Final LSTM layer without return_sequences
        model.add(layers.LSTM(self.units[-1], activation='tanh', 
                              kernel_regularizer=tf.keras.regularizers.l2(self.kernel_regularizer_l2)))
        model.add(layers.BatchNormalization())
        model.add(layers.Dropout(self.dropout_ratio))

        # Final output layer
        model.add(layers.Dense(1, activation='linear'))  # Use sigmoid activation

        # Use a custom output layer to scale the output to 1-9
        # model.add(layers.Lambda(lambda x: x * 8 + 1)) 
        # model.add(layers.Lambda(lambda x: x * 10)) 

        model.compile(
            optimizer = keras.optimizers.AdamW(
                learning_rate=self.learning_rate,
                weight_decay=self.weight_decay,
                # beta_1=self.beta_1,
                # beta_2=self.beta_2,
                # epsilon=self.epsilon
            ),
            loss='mse',
            metrics=['mae']
        )

        return model



    # First, define the train_single_fold function outside of the class
    def _train_single_fold(self, args):
        """Helper function for parallel processing in test_model.

        Args:
            args: Tuple containing (model_instance, test_idx, val_idx, data)

        Returns:
            Dictionary containing model, history, and reports
        """
        test_subject_idx, data = args
        local_results = {}

        subject_fold = data[data['subject_id']==test_subject_idx]['fold'].values[0]
        validation_subjects_idx = data[(data['subject_id']!=test_subject_idx) & (data['fold']==subject_fold)]['subject_id'].unique().tolist()
        logging.info(f"Starting Test {test_subject_idx} & Validation {validation_subjects_idx}")

        train = data[~data['subject_id'].isin(validation_subjects_idx + [test_subject_idx])]
        val   = data[ data['subject_id'].isin(validation_subjects_idx)]
        test  = data[ data['subject_id']==test_subject_idx]
        logging.info(f"train {train.shape} & val {val.shape} & test {test.shape}")

        # List of columns to exclude from scaling
        exclude_cols    = ['slice_index', 'filename', 'start_frame', 'end_frame', 'fold', 'subject_id', 'target']
        
        train = train.drop(columns=['fold', 'subject_id', 'split'])
        val   = val.drop(  columns=['fold', 'subject_id', 'split'])
        test  = test.drop( columns=['fold', 'subject_id', 'split'])
        featur_columns = [c for c in train.columns if c.startswith('pc')]
        
        X_train, y_train = reshape_data_rnn(train, featur_columns, max_slices=self.max_slices, sortby_column='frame', target_column='target')
        X_val,   y_val   = reshape_data_rnn(val,   featur_columns, max_slices=self.max_slices, sortby_column='frame', target_column='target')
        X_test,  y_test  = reshape_data_rnn(test,  featur_columns, max_slices=self.max_slices, sortby_column='frame', target_column='target')

        logging.info(f"X_train {X_train.shape} & X_val {X_val.shape} & X_test {X_test.shape}")

        sample_weight = compute_sample_weight(class_weight='balanced', y=y_train)
        
        logging.info(f"model")
        model = self.build_model((X_train.shape[1], X_train.shape[2]))

        logging.info(f"fit")
        # Train the model
        history = model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            epochs=self.epochs,
            batch_size=self.batch_size,
            callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=self.patience, restore_best_weights=True)],
            verbose=0,  # Reduced verbosity for parallel execution
            sample_weight=sample_weight
        )
        
        logging.info(f"rep")
        y_train_pred, train_report = report(model=model, x=X_train, y_true=y_train, unique_classes=list(self.mapping.values()))
        y_val_pred,   val_report   = report(model=model, x=X_val,   y_true=y_val,   unique_classes=list(self.mapping.values()))
        y_test_pred,  test_report  = report(model=model, x=X_test,  y_true=y_test,  unique_classes=list(self.mapping.values()))

        # Store results to return
        local_results = {
            'test': test_subject_idx,
            'val': validation_subjects_idx,
            'history': history.history,
            'train_report': train_report,
            'y_train': y_train,
            'y_train_pred': y_train_pred,
            'val_report': val_report,
            'y_val': y_val,
            'y_val_pred': y_val_pred,
            'test_report': test_report,
            'y_test': y_test,
            'y_test_pred': y_test_pred,
        }
        
        logging.info(f"epoch : {len(history.history['val_loss'])}")
        logging.info(f"Completed Test {test_subject_idx} & Validation {validation_subjects_idx}")
        logging.info('=======================================')

        return local_results
            

    def test_model(self, data):
        unique_subject_id = data['subject_id'].unique()

        logging.info(f"Found {len(unique_subject_id)} valid test-validation pairs.")

        # Prepare arguments for parallel processing
        args_list = [(i, data) for i in unique_subject_id]
        random.shuffle(args_list)

        output_folder_name = datetime.now().strftime("%Y-%m-%d|%H:%M:%S|RNNPCA_MNV2_1000")
        output_parent_dir = "/home/mojtaba.kananisarche/Zh/UTA/Output/"
        output_folder_path = os.path.join(output_parent_dir, output_folder_name)
        os.makedirs(output_folder_path, exist_ok=True)

        with open(os.path.join(output_folder_path, 'results.json'), 'w') as file:
            json.dump({
                'general':{
                    'units':self.units,
                    'dropout_ratio':self.dropout_ratio,
                    'batch_size':self.batch_size,
                    'learning_rate':self.learning_rate,
                    'weight_decay':self.weight_decay,
                    'epochs':self.epochs,
                    'dataset_id':self.dataset_id,
                    'max_slices':self.max_slices
                }
            }, file)

        # Run training in parallel and process results as they come
        all_results = []

        # with concurrent.futures.ProcessPoolExecutor(max_workers=10) as executor:
        #     futures = [executor.submit(self._train_single_fold, args) for args in args_list]
        #     for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Training folds"):
        #         result = future.result()
        #         file_path = os.path.join(output_folder_path, f'{result["test"]}.json')
        #         with open(file_path, 'w') as file:
        #             json.dump(result, file, cls=NumpyEncoder, indent=4)
        #             logging.info(f"Saved results for Test subject {result['test']} to {file_path}")

        for args in args_list:
            result = self._train_single_fold(args)
            file_path = os.path.join(output_folder_path, f'{result["test"]}.json')
            with open(file_path, 'w') as file:
                json.dump(result, file, cls=NumpyEncoder, indent=4)
                logging.info(f"Saved results for Test subject {result['test']} to {file_path}")
                
                    
        logging.info(f"Completed all {len(unique_subject_id)} test-validation pairs.")

In [4]:
logging.info(f"Starting")
# Call the function to set the seed
set_seed(42)

mapping = {
    '10' : 8.5,
    '5' : 6.5,
    '0' : 2
}

dataset_name = "pca_transformed_test_validation_2.csv"
dataset_dir  = "/home/mojtaba.kananisarche/Zh/UTA/feature_vectors/mobilenetv2/pca_transformed/"
dataset_path = os.path.join(dataset_dir, dataset_name)
df = pd.read_csv(dataset_path)
df['target'] = df['target'].astype(str)
df['target'] = df['filename'].apply(lambda x:mapping.get(x.split('_')[2]))



m = Model(units = [100, 50], 
        dropout_ratio = 0.1, 
        kernel_regularizer_l2 = 0, 
        kernel_regularizer_l1 = 0, 
        batch_size = 16, 
        num_classes = 3, 
        learning_rate = 1e-3, 
        weight_decay = 1e-4, 
        beta_1 = 0.9, 
        beta_2 = 0.999, 
        epsilon = 1e-7,
        epochs = 1, 
        patience = 100,
        max_slices=3000,
        mapping = mapping,
        dataset_name = dataset_name)

m.test_model(df)

2025-03-26 07:47:21,726 - INFO - Starting
2025-03-26 07:48:53,910 - INFO - Found 60 valid test-validation pairs.
2025-03-26 07:48:53,928 - INFO - Note: detected 80 virtual cores but NumExpr set to maximum of 64, check "NUMEXPR_MAX_THREADS" environment variable.
2025-03-26 07:48:53,929 - INFO - Note: NumExpr detected 80 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 8.
2025-03-26 07:48:53,929 - INFO - NumExpr defaulting to 8 threads.
2025-03-26 07:48:56,161 - INFO - Starting Test 58 & Validation [52, 49, 54, 53, 56, 57, 51, 50, 59, 55, 60]
2025-03-26 07:48:57,505 - INFO - train (2566833, 179) & val (651484, 179) & test (54751, 179)


KeyboardInterrupt: 

In [28]:
def plot(df):
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    import matplotlib.gridspec as gridspec

    # Load your dataset
    # df = pd.read_csv('your_dataset.csv')  # Uncomment if needed

    # Unique classes and colors
    classes = sorted(df['target'].unique())
    cmap = plt.get_cmap('tab10')
    colors = [cmap(i) for i in range(len(classes))]

    # Create figure and layout
    fig = plt.figure(figsize=(10, 10))
    gs = gridspec.GridSpec(4, 4)

    # Scatter plot (main panel)
    ax_main = fig.add_subplot(gs[1:4, 0:3])

    # Top distribution (PC1)
    ax_xdist = fig.add_subplot(gs[0, 0:3], sharex=ax_main)

    # Right distribution (PC2)
    ax_ydist = fig.add_subplot(gs[1:4, 3], sharey=ax_main)

    # Plot scatter for each class
    for i, cls in enumerate(classes):
        cls_data = df[df['target'] == cls]
        
        # Scatter
        ax_main.scatter(
            cls_data['PC1'], cls_data['PC2'],
            color=colors[i],
            alpha=0.6,
            edgecolors='k',
            label=f'Class {cls}',
            s=30
        )
        
        # KDE for PC1 (top)
        sns.kdeplot(cls_data['PC1'], ax=ax_xdist, color=colors[i], label=f'Class {cls}', fill=True, alpha=0.3)
        
        # KDE for PC2 (right)
        sns.kdeplot(cls_data['PC2'], ax=ax_ydist, color=colors[i], fill=True, alpha=0.3, vertical=True)

    # Hide x-axis labels on top and y-axis labels on right
    ax_xdist.set_ylabel('Density')
    ax_ydist.set_xlabel('Density')
    
    # Create a custom legend in the top-right corner of the figure
    handles, labels = ax_main.get_legend_handles_labels()
    fig.legend(handles, labels, title='Target Class', loc='upper right', bbox_to_anchor=(0.98, 0.98))

    ax_xdist.grid(True)
    ax_ydist.grid(True)

    # Set main scatter plot labels
    ax_main.set_xlabel('PC1')
    ax_main.set_ylabel('PC2')
    ax_main.grid(True)
    ax_main.set_title('PC1 vs PC2 Scatter Plot with Class Distributions')

    # Tidy layout
    plt.tight_layout()
    plt.savefig('./pc1_pc2_class_distribution_plot_mnv2_unscaled.png', dpi=300)
    plt.close()


In [3]:
dataset_name = "scaled_test_validation_1.csv"
dataset_dir  = "/home/mojtaba.kananisarche/Zh/UTA/feature_vectors/mobilenetv2/standard_scaled/"
dataset_path = os.path.join(dataset_dir, dataset_name)
df = pd.read_csv(dataset_path)

In [26]:
temp_df = df.loc[:, ['pc_0', 'pc_1', 'target']]
temp_df.columns = ['PC1', 'PC2', 'target']
temp_df
# plot(temp_df)

,PC1,PC2,target
0,0.376867,-0.652122,10
1,-0.094885,-0.894004,10
2,0.412897,-0.591778,10
3,0.534159,-0.714016,10
4,0.176389,-0.599530,10
...,...,...,...
3273063,2.306501,1.546500,10
3273064,2.186084,1.556121,10
3273065,2.107244,1.748425,10
3273066,2.137327,1.532426,10


In [23]:
dataset_name = "pca_transformed_test_validation_1.csv"
dataset_dir  = "/home/mojtaba.kananisarche/Zh/UTA/feature_vectors/mobilenetv2/pca_transformed/"
dataset_path = os.path.join(dataset_dir, dataset_name)
df2 = pd.read_csv(dataset_path)

In [27]:
temp_df = df2.loc[:, ['pc_0', 'pc_1', 'target']]
temp_df.columns = ['PC1', 'PC2', 'target']
temp_df
# plot(temp_df)

,PC1,PC2,target
0,3.314612,-5.317406,10
1,-0.834529,-7.289718,10
2,3.631508,-4.825364,10
3,4.698029,-5.822088,10
4,1.551375,-4.888574,10
...,...,...,...
3273063,20.286104,12.610172,10
3273064,19.227011,12.688623,10
3273065,18.533602,14.256671,10
3273066,18.798186,12.495409,10


In [29]:
plot(temp_df)

/opt/conda/lib/python3.8/site-packages/seaborn/distributions.py:1647: FutureWarning: The `vertical` parameter is deprecated and will be removed in a future version. Assign the data to the `y` variable instead.
  warnings.warn(msg, FutureWarning)
/opt/conda/lib/python3.8/site-packages/seaborn/distributions.py:1647: FutureWarning: The `vertical` parameter is deprecated and will be removed in a future version. Assign the data to the `y` variable instead.
  warnings.warn(msg, FutureWarning)
/opt/conda/lib/python3.8/site-packages/seaborn/distributions.py:1647: FutureWarning: The `vertical` parameter is deprecated and will be removed in a future version. Assign the data to the `y` variable instead.
  warnings.warn(msg, FutureWarning)


In [ ]:
logging.info(f"Starting")
# Call the function to set the seed
set_seed(42)

dataset_name = "scaled_test_validation_1.csv"
dataset_dir  = "/home/mojtaba.kananisarche/Zh/UTA/feature_vectors/mobilenetv2/standard_scaled/"
dataset_path = os.path.join(dataset_dir, dataset_name)
df = pd.read_csv(dataset_path, index_col=0)

mapping = {
    '10' : 8.5,
    '5' : 6.5,
    '0' : 2
}


m = Model(units = [20], 
        dropout_ratio = 0.3, 
        kernel_regularizer_l2 = 0, 
        kernel_regularizer_l1 = 0, 
        batch_size = 32, 
        num_classes = 3, 
        learning_rate = 1e-3, 
        weight_decay = 1e-4, 
        beta_1 = 0.9, 
        beta_2 = 0.999, 
        epsilon = 1e-7,
        epochs = 1000, 
        patience = 100,
        max_slices=50,
        mapping = mapping,
        dataset_name = dataset_name)

m.test_model(df)
